# 1. Linear expressions and constraints in Pyomo

Lecture PDF pages 5–6: *Linear Expressions in Pyomo* and *Linear Constraints in Pyomo*.

**What it shows.** How Pyomo builds a linear expression such as $10x_1 + 6x_2 + 8x_3 + 16x_4$, either written
out term by term or with `sum(...)` over a list of coefficients (the matrix form $a^T x = \sum_i a_i x_i$ of
page 4); how to turn it into a constraint and an objective; and how to print, replace and switch off a
constraint. Nothing is solved here. Python counts from 0, so the slides' $x_1, \dots, x_4$ are `x[0]` … `x[3]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** The constraint $10x_1 + 6x_2 + 8x_3 + 16x_4 \ge 120$ (as on page 6), the same expression
printed three more times, the objective, a Pyomo **WARNING** (explained below), the constraint again with 32 as
its lower limit, and a weight constraint shown first as `Active : True` and then as `False`.

*Original script: `5 Source Files/Python/pyomo_printing_demo.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for variables, constraints and the objective.

In [ ]:
# Create a Concrete Model
model = ConcreteModel()

This cell does several things in turn: it creates four variables, builds $10x_1 + 6x_2 + 8x_3 + 16x_4 \ge 120$ as a constraint and prints it (page 6), prints the same expression written out by hand and then with `sum(...)` (page 5: both give exactly the same expression), and finally uses the expression as an objective to maximise.

In [ ]:
# Four continuous (real) variables x[0]..x[3], each >= 0 (NonNegativeReals, not integers)
model.x = Var(range(4), domain=NonNegativeReals)
# the coefficients a = (10, 6, 8, 16) as a Python list
cost = [10, 6, 8, 16]
# a linear inequality constraint: 10x1 + 6x2 + 8x3 + 16x4 >= 120
model.constraint1 = Constraint(expr=sum(cost[i] * model.x[i] for i in model.x) >= 120)
# pretty print the constraint: lower limit 120, upper limit +Inf
model.constraint1.pprint()


print(10*model.x[0] + 6*model.x[1] + 8*model.x[2] + 16*model.x[3])  # written out term by term

print()


print(sum(cost[i] * model.x[i] for i in model.x))  # the same expression with sum(...)


volume = [100, 60, 85, 160]  # not used below (left over from the original script)
weight = [1.05, 2.6, 3.8, 1.6]  # weights, used for constraint2 below

print(sum(cost[i] * model.x[i] for i in model.x))  # printed once more (the original script repeats it)
print()
# Objective: maximise the same expression
model.obj = Objective(expr=sum(cost[i] * model.x[i] for i in model.x), sense=maximize)
model.obj.pprint()

print()

Defining `model.constraint1` a second time **replaces** the first one (now with a lower limit of 32). Pyomo allows this but prints *WARNING: Implicitly replacing the Component attribute constraint1* to tell you that a constraint was overwritten. Here it is on purpose; in your own models the warning usually means a name was reused by mistake. (To replace a constraint without the warning, call `model.del_component(model.constraint1)` first.)

In [ ]:
# Redefine constraint1 with a new lower limit (>= 32): this triggers the WARNING
model.constraint1 = Constraint(expr=sum(cost[i] * model.x[i] for i in model.x) >= 32)
model.constraint1.pprint()

print()

A second constraint, this time a $\le$ one: total weight $1.05x_1 + 2.6x_2 + 3.8x_3 + 1.6x_4 \le 100$. In the printout the lower limit is `-Inf` and the upper limit is 100.

In [ ]:
# Define a <= constraint on total weight
model.constraint2 = Constraint(expr=sum(weight[i] * model.x[i] for i in model.x) <= 100)
model.constraint2.pprint()

`deactivate()` switches a constraint off without deleting it: the printout now says `Active : False`, and a solver would ignore it. `model.constraint2.activate()` switches it back on.

In [ ]:
# switch the constraint off (it stays in the model; activate() switches it back on)
model.constraint2.deactivate()
model.constraint2.pprint()

## Try it

1. Change `>= 120` in the first constraint to `<= 120` and run the cells again. What changes in its printout?
2. Use `sum(...)` with the `weight` list to build and print $1.05x_1 + 2.6x_2 + 3.8x_3 + 1.6x_4$.

<details><summary>Answer</summary>

1. The `Lower` column becomes `-Inf` and the `Upper` column `120.0`: Pyomo stores every constraint as
   lower ≤ body ≤ upper, with `±Inf` for a side that has no limit.
2. `print(sum(weight[i] * model.x[i] for i in model.x))` prints `1.05*x[0] + 2.6*x[1] + 3.8*x[2] + 1.6*x[3]`, the
   same body as `constraint2`.
</details>